# 台股廣度指標 — 冷啟動（歷史資料初始化）

使用 **Yahoo Finance** 抓取資料，完全免費，不需要任何 Token。

執行這個筆記本會：
1. 從 TWSE + TPEX 官方 API 取得個股清單
2. 用 yfinance 批次抓取所有個股兩年日K
3. 計算每日廣度指標
4. 產生 `breadth_ma.json` 與 `breadth_52week.json`
5. 自動下載這兩個檔案 → 上傳到 GitHub repo 的 `data/` 資料夾

⚠️ **不需要填入任何 Token，直接從第一個 Cell 開始執行即可**

In [ ]:
# Cell 1：安裝套件
!pip install yfinance requests pandas -q
print('套件安裝完成')

In [ ]:
# Cell 2：匯入套件與定義工具函式
import yfinance as yf
import requests
import pandas as pd
import json
import time
import logging
from datetime import datetime, timedelta

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
log = logging.getLogger(__name__)

FETCH_DAYS         = 730   # 抓 2 年，確保 200MA 資料足夠
TRADING_DAYS_200MA = 200
TRADING_DAYS_52W   = 252

print('函式定義完成')

In [ ]:
# Cell 3：從 TWSE + TPEX 取得個股清單

def get_twse_stocks():
    """取得上市普通股清單"""
    url = 'https://openapi.twse.com.tw/v1/exchangeReport/STOCK_DAY_ALL'
    try:
        r = requests.get(url, timeout=30)
        data = r.json()
        df = pd.DataFrame(data)
        # 只保留 4 碼純數字（排除 ETF 等）
        df = df[df['Code'].str.match(r'^\d{4}$')]
        df = df[['Code', 'Name']].copy()
        df.columns = ['stock_id', 'stock_name']
        df['suffix'] = '.TW'
        return df
    except Exception as e:
        log.error('TWSE API 失敗: %s', e)
        return pd.DataFrame()

def get_tpex_stocks():
    """取得上櫃普通股清單"""
    url = 'https://www.tpex.org.tw/openapi/v1/tpex_mainboard_quotes'
    try:
        r = requests.get(url, timeout=30)
        data = r.json()
        df = pd.DataFrame(data)
        # 只保留 4 碼純數字
        df = df[df['SecuritiesCompanyCode'].str.match(r'^\d{4}$')]
        df = df[['SecuritiesCompanyCode', 'CompanyName']].copy()
        df.columns = ['stock_id', 'stock_name']
        df['suffix'] = '.TWO'
        return df
    except Exception as e:
        log.error('TPEX API 失敗: %s', e)
        return pd.DataFrame()

twse = get_twse_stocks()
tpex = get_tpex_stocks()
stocks = pd.concat([twse, tpex], ignore_index=True)

# 產生 Yahoo Finance 代號：2330.TW 或 6488.TWO
stocks['yf_symbol'] = stocks['stock_id'] + stocks['suffix']

print(f'上市：{len(twse)} 筆')
print(f'上櫃：{len(tpex)} 筆')
print(f'合計：{len(stocks)} 筆普通股')
stocks.head(10)

In [ ]:
# Cell 4：設定日期區間
end_date   = datetime.now().strftime('%Y-%m-%d')
start_date = (datetime.now() - timedelta(days=FETCH_DAYS)).strftime('%Y-%m-%d')
print(f'抓取區間：{start_date} ~ {end_date}')
print(f'總計 {len(stocks)} 支個股')
print(f'yfinance 批次下載，預計約 10~20 分鐘')

In [ ]:
# Cell 5：批次抓取所有個股收盤價（yfinance 一次抓多支，速度快）
symbols = stocks['yf_symbol'].tolist()
BATCH_SIZE = 100  # 每批 100 支

all_close = {}
failed_symbols = []

for i in range(0, len(symbols), BATCH_SIZE):
    batch = symbols[i:i+BATCH_SIZE]
    batch_str = ' '.join(batch)
    try:
        raw = yf.download(
            batch_str,
            start=start_date,
            end=end_date,
            auto_adjust=True,
            progress=False,
            threads=True,
        )
        # yfinance 多支股票時回傳 MultiIndex，取 Close 欄
        if isinstance(raw.columns, pd.MultiIndex):
            close_df = raw['Close']
        else:
            close_df = raw[['Close']]
            close_df.columns = batch

        for sym in batch:
            if sym in close_df.columns:
                s = close_df[sym].dropna()
                if len(s) > 50:  # 至少要有 50 天資料才納入
                    all_close[sym] = s
                else:
                    failed_symbols.append(sym)
            else:
                failed_symbols.append(sym)

    except Exception as e:
        log.warning('批次 %d 失敗: %s', i//BATCH_SIZE+1, e)
        failed_symbols.extend(batch)

    done = min(i+BATCH_SIZE, len(symbols))
    print(f'進度：{done}/{len(symbols)}，成功 {len(all_close)}，失敗 {len(failed_symbols)}')
    time.sleep(1)  # 避免過快

print(f'\n抓取完成！成功 {len(all_close)} 支，失敗 {len(failed_symbols)} 支')

In [ ]:
# Cell 6：計算廣度指標
price_df = pd.DataFrame(all_close).sort_index()
print(f'合併後：{price_df.shape[0]} 個交易日 × {price_df.shape[1]} 支個股')

chart_start  = pd.Timestamp(end_date) - pd.DateOffset(years=1)
output_dates = price_df.index[price_df.index >= chart_start]
print(f'輸出區間：{output_dates[0].date()} ~ {output_dates[-1].date()}，共 {len(output_dates)} 個交易日')

ma_records     = []
week52_records = []

for n, dt in enumerate(output_dates):
    loc = price_df.index.get_loc(dt)

    # ── MA
    w50         = price_df.iloc[max(0, loc-49):loc+1]
    w200        = price_df.iloc[max(0, loc-199):loc+1]
    close_today = price_df.loc[dt]
    ma50        = w50.mean()
    ma200       = w200.mean()
    valid       = close_today.notna() & ma50.notna() & ma200.notna()
    ma_records.append({
        'date':       dt.strftime('%Y-%m-%d'),
        'above50ma':  int((close_today[valid] > ma50[valid]).sum()),
        'above200ma': int((close_today[valid] > ma200[valid]).sum()),
        'total':      int(valid.sum()),
    })

    # ── 52週
    w52w = price_df.iloc[max(0, loc-251):loc+1]
    h52w = w52w.max()
    l52w = w52w.min()
    v52  = close_today.notna() & h52w.notna() & l52w.notna()
    week52_records.append({
        'date':    dt.strftime('%Y-%m-%d'),
        'high52w': int((close_today[v52] >= h52w[v52]).sum()),
        'low52w':  int((close_today[v52] <= l52w[v52]).sum()),
        'total':   int(v52.sum()),
    })

    if (n+1) % 50 == 0:
        print(f'計算進度：{n+1}/{len(output_dates)}')

print(f'\n計算完成！{len(ma_records)} 筆 MA 資料，{len(week52_records)} 筆 52週資料')

In [ ]:
# Cell 7：儲存並下載 JSON
from google.colab import files

with open('breadth_ma.json', 'w', encoding='utf-8') as f:
    json.dump(ma_records, f, ensure_ascii=False, indent=2)

with open('breadth_52week.json', 'w', encoding='utf-8') as f:
    json.dump(week52_records, f, ensure_ascii=False, indent=2)

print('檔案已產生，即將下載...')
files.download('breadth_ma.json')
files.download('breadth_52week.json')
print('下載完成！請將這兩個檔案上傳到 GitHub repo 的 data/ 資料夾')